# 1. Depth-cache coverage and eddy-spine displacement

Before comparing physics, this notebook visualises which Eddy–Day observations survive at each cached depth. Formal comparisons use only observations available at every selected level, preventing depth-dependent sample attrition from masquerading as a physical signal.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import seacofs_tilt_tools as tilt
import depth_pv_tools as dpt

sns.set_theme(style="whitegrid", context="notebook")
DOMINANCE_FACTOR = 2.0
TARGET_DEPTHS_M = (0, 200, 500, 700, 1000)
MIN_TILT_KM = 5.0

depth_df = tilt.add_pv_gradient_terms(source="depth")
snapshot_df = tilt.add_pv_gradient_terms(source="depth_snapshot")
dpt.validate_depth_tables(depth_df, snapshot_df)
DEPTHS = dpt.nearest_cached_depths(depth_df, TARGET_DEPTHS_M)
comparison = dpt.add_surface_differences(depth_df, DOMINANCE_FACTOR)
matched = dpt.matched_depth_rows(comparison, DEPTHS)
palette = {"AE": "#c44e52", "CE": "#4c72b0"}
depth_cmap = plt.get_cmap("viridis")
depth_colours = dict(zip(DEPTHS, depth_cmap(np.linspace(.08, .92, len(DEPTHS)))))
depth_labels = {z: f"{z:g} m" for z in DEPTHS}


In [ ]:
coverage = (depth_df.groupby("Depth")
            .agg(snapshots=("Day", "size"), eddies=("Eddy", "nunique"))
            .reset_index())
surface_n = coverage.snapshots.iloc[0]
coverage["snapshot_fraction"] = coverage.snapshots / surface_n

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(coverage.snapshot_fraction, coverage.Depth, color="black", lw=2)
axes[0].scatter(coverage.snapshot_fraction, coverage.Depth, c=coverage.Depth, cmap="viridis", s=25)
axes[0].invert_yaxis(); axes[0].set(xlabel="Fraction of shallow-level snapshots retained", ylabel="Cached depth (m)", xlim=(0, 1.03))
axes[1].plot(coverage.eddies, coverage.Depth, color="tab:purple", lw=2)
axes[1].invert_yaxis(); axes[1].set(xlabel="Unique eddies", ylabel="Cached depth (m)")
fig.suptitle("Depth coverage of the PV-gradient cache")
plt.show()

In [ ]:
availability = (depth_df.assign(present=1)
                .pivot_table(index=["Cyc", "Eddy"], columns="Depth", values="present", aggfunc="max", fill_value=0))
order = availability.sum(axis=1).sort_values(ascending=False).index
show = availability.loc[order]
fig, ax = plt.subplots(figsize=(13, 7), constrained_layout=True)
ax.imshow(show.to_numpy(), aspect="auto", interpolation="nearest", cmap="Greys", vmin=0, vmax=1)
ticks = np.linspace(0, len(show.columns)-1, min(8, len(show.columns))).astype(int)
ax.set(xticks=ticks, xticklabels=[f"{show.columns[i]:g}" for i in ticks], xlabel="Depth (m)", ylabel="Eddies (ordered by coverage)", title="Available depth levels for every eddy")
plt.show()

In [ ]:
eddy_disp = (comparison.groupby(["Cyc", "Depth", "Eddy"], observed=True).spine_displacement_km.median().reset_index())
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
sns.lineplot(data=eddy_disp, x="spine_displacement_km", y="Depth", hue="Cyc", palette=palette,
             estimator="median", errorbar=("pi", 50), marker="o", ax=ax)
ax.invert_yaxis(); ax.set(xlabel="Median displacement from shallow centre (km)", ylabel="Depth (m)", title="Horizontal separation grows down the eddy spine")
plt.show()

## Reading this notebook

Use the coverage figures to decide whether the deepest selected level is sufficiently represented. The paired notebooks deliberately use the matched sample (`matched`) rather than all available rows.